# 01 — EIA operable-generator pull (date pinned)

Reinstated from the last committed pre-pivot notebook for the provenance repair. The EIA request is constrained to the most recent complete monthly inventory available at retrieval time: **2026-05**. Every response page is archived in `data/staging/` with retrieval metadata and SHA-256.


In [1]:
from pathlib import Path
import hashlib
import json
import sys
from datetime import datetime, timezone

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import utils

STAGING = PROJECT_ROOT / "data" / "staging"
STAGING.mkdir(parents=True, exist_ok=True)

ENDPOINT = "electricity/operating-generator-capacity/data"
PINNED_PERIOD = "2026-05"
PAGE_SIZE = 5000
BASE_PARAMS = {
    "frequency": "monthly",
    "start": PINNED_PERIOD,
    "end": PINNED_PERIOD,
    "data[0]": "nameplate-capacity-mw",
    "data[1]": "latitude",
    "data[2]": "longitude",
    "data[3]": "county",
    "facets[status][0]": "OP",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "length": PAGE_SIZE,
}

pages = []
records = []
offset = 0
expected_total = None
while expected_total is None or offset < expected_total:
    response = utils.eia_get(ENDPOINT, {**BASE_PARAMS, "offset": offset})
    block = response.get("response", {})
    page_records = block.get("data", [])
    if expected_total is None:
        expected_total = int(block["total"])
    if not page_records:
        raise RuntimeError(f"EIA pagination ended at {offset:,} before expected total {expected_total:,}")
    response.get("request", {}).get("params", {}).pop("api_key", None)
    pages.append(response)
    records.extend(page_records)
    offset += len(page_records)

periods = {row.get("period") for row in records}
if periods != {PINNED_PERIOD}:
    raise RuntimeError(f"EIA period pin failed: expected {PINNED_PERIOD}, received {sorted(periods)}")
if len(records) != expected_total:
    raise RuntimeError(f"EIA record count mismatch: {len(records):,} != {expected_total:,}")

raw_path = STAGING / f"eia_operating_generators_{PINNED_PERIOD}_raw.json"
raw_bytes = json.dumps(
    {"endpoint": ENDPOINT, "params_without_api_key": BASE_PARAMS, "pages": pages},
    separators=(",", ":"),
    sort_keys=True,
).encode()
raw_path.write_bytes(raw_bytes)

df = pd.DataFrame(records)
required = {"latitude", "longitude", "nameplate-capacity-mw"}
missing = required.difference(df.columns)
if missing:
    raise KeyError(f"EIA response missing required columns: {sorted(missing)}")
gdf = utils.df_to_geodataframe(df, lat_col="latitude", lon_col="longitude")
gdf.to_file(STAGING / "power_plants.geojson", driver="GeoJSON")

metadata = {
    "source": "U.S. EIA API v2, Inventory of Operable Generators",
    "endpoint": f"https://api.eia.gov/v2/{ENDPOINT}",
    "pinned_period": PINNED_PERIOD,
    "status_filter": "OP",
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "raw_response_path": str(raw_path.relative_to(PROJECT_ROOT)),
    "raw_sha256": hashlib.sha256(raw_bytes).hexdigest(),
    "raw_bytes": len(raw_bytes),
    "record_count": len(records),
    "derived_path": "data/staging/power_plants.geojson",
}
(STAGING / "eia_operating_generators_2026-05_metadata.json").write_text(
    json.dumps(metadata, indent=2) + "\n"
)
print(json.dumps(metadata, indent=2))


{
  "source": "U.S. EIA API v2, Inventory of Operable Generators",
  "endpoint": "https://api.eia.gov/v2/electricity/operating-generator-capacity/data",
  "pinned_period": "2026-05",
  "status_filter": "OP",
  "retrieved_at_utc": "2026-08-10T21:00:59.046049+00:00",
  "raw_response_path": "data/staging/eia_operating_generators_2026-05_raw.json",
  "raw_sha256": "cf14ff4b1283ca3c7ad563d93ba098ae8a761561c2396d58252aab4343179a36",
  "raw_bytes": 16888632,
  "record_count": 25868,
  "derived_path": "data/staging/power_plants.geojson"
}
